# UAV-SEAD probabilistic GPU v2 — balanced Colab L4

Bu notebook yeni four_dataset_probabilistic_gpu_v2 sözleşmesini çalıştırır.
Tamamlanmış v1 sonucu değiştirilmez. Model geçmiş 32 satırdan bir sonraki
satırın kanal bazlı mean ve scale değerlerini öğrenir; optimizer yalnız normal
uçuşları görür.

Frozen source-level roller:

- normal train: 629
- normal validation: 135
- primary test: 268 kaynak
- primary test normal/anomaly: 134/134
- ayrı stress-test anomaly havuzu: 212

Normal havuz 70/15/15 bölünmüştür. Ana test, sınıf oranının ROC/AP yorumunu
bozmaması için dengelenmiştir. Stress-test havuzu threshold, optimizer veya
primary sonuç seçiminde kullanılmaz.


## 1. Drive ve sabit yollar

Mevcut four_dataset_probabilistic_v1_transfer klasöründeki dataset ZIP'i yeniden kullanılabilir. Aynı klasöre yalnız four_dataset_probabilistic_gpu_v2_code_and_contract.zip, transfer_index_v2.json ve bundle_manifest_v2.json dosyalarını ekleyin. Eski transfer_index.json dosyasını silmeyin; bu notebook yalnız transfer_index_v2.json dosyasını okur.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATASET = 'uav_sead'
TRANSFER_DIR = Path('/content/drive/MyDrive/bykr/four_dataset_probabilistic_v1_transfer')
REPO_ROOT = Path('/content/four_dataset_probabilistic_gpu_v2')
RUN_DIR = Path('/content/drive/MyDrive/bykr/four_dataset_probabilistic_v2_runs') / f'{DATASET}_gpu_v2'
REPO_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not TRANSFER_DIR.is_dir():
    raise FileNotFoundError(f'Transfer klasörü yok: {TRANSFER_DIR}')
print('Dataset:', DATASET)
print('Repo:', REPO_ROOT, '| Run:', RUN_DIR)

## 2. CUDA/L4 doğrulaması

CUDA yoksa fail-loudly durur ve CPU fallback yapılmaz. L4 önerilen profildir; T4/A100 gibi başka bir CUDA GPU yalnız uyarı üretir.

In [ ]:
import platform
import torch
print('Python:', platform.python_version(), '| Torch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA yok; Colab L4 GPU runtime seçin.')
gpu_name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print('GPU:', gpu_name, '| VRAM GiB:', round(vram_gib, 2))
if 'L4' not in gpu_name.upper():
    print(f'UYARI: Önerilen L4 yerine {gpu_name} kullanılıyor; CUDA contract geçerli.')

## 3. Kod ve dataset paketlerini aç

SHA-256 marker aynı runtime'da tekrar açmayı önler; değişen paket yeniden açılır. Güvensiz ZIP yolları reddedilir.

In [ ]:
import hashlib
import json
import zipfile
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(8 * 1024 * 1024)
            if not block:
                return digest.hexdigest()
            digest.update(block)
index_path = TRANSFER_DIR / 'transfer_index_v2.json'
if not index_path.is_file():
    raise RuntimeError(f'CONTRACT ERROR: transfer_index_v2.json yok: {index_path}')
expected_index_sha256 = '1248aef88f790f102cec5aad297a82f33779d4c7e85b03c03a2733b664fb23e9'
observed_index_sha256 = sha256_file(index_path)
if observed_index_sha256 != expected_index_sha256:
    raise RuntimeError(f'CONTRACT ERROR: transfer_index_v2.json SHA-256 uyuşmazlığı: {observed_index_sha256}')
print('INDEX SHA PASS:', observed_index_sha256[:12])
transfer_index = json.loads(index_path.read_text(encoding='utf-8'))
if transfer_index.get('candidate_namespace') != 'four_dataset_probabilistic_gpu_v2':
    raise RuntimeError('CONTRACT ERROR: transfer index namespace uyuşmuyor')
if transfer_index.get('archives_built') is not True:
    raise RuntimeError('CONTRACT ERROR: transfer index built archive bildirmiyor')
records = transfer_index.get('archives')
if not isinstance(records, list):
    raise RuntimeError('CONTRACT ERROR: transfer index archives listesi yok')
code_matches = [r for r in records if 'code_and_contract' in str(r.get('path', ''))]
data_matches = [r for r in records if DATASET in str(r.get('path', '')) and r not in code_matches]
if len(code_matches) != 1 or len(data_matches) != 1:
    raise RuntimeError(f'CONTRACT ERROR: archive seçimi tekil değil: code={code_matches}, data={data_matches}')
def verify_record(record):
    name = record.get('path')
    expected_sha = record.get('sha256')
    expected_bytes = record.get('bytes')
    if not isinstance(name, str) or Path(name).name != name:
        raise RuntimeError(f'CONTRACT ERROR: güvensiz archive adı: {name!r}')
    if not isinstance(expected_sha, str) or len(expected_sha) != 64 or not isinstance(expected_bytes, int):
        raise RuntimeError(f'CONTRACT ERROR: hash/byte kaydı eksik: {name}')
    archive = TRANSFER_DIR / name
    if not archive.is_file() or archive.stat().st_size != expected_bytes:
        raise RuntimeError(f'CONTRACT ERROR: archive yok veya byte boyutu yanlış: {archive}')
    observed = sha256_file(archive)
    if observed != expected_sha:
        raise RuntimeError(f'CONTRACT ERROR: SHA-256 uyuşmazlığı: {archive}')
    print('SHA PASS:', name, observed[:12])
    return archive, observed
verified = [verify_record(code_matches[0]), verify_record(data_matches[0])]
CODE_ZIP, DATA_ZIP = verified[0][0], verified[1][0]
def extract_verified(archive, observed, destination):
    markers = destination / '.colab_extraction_markers'
    markers.mkdir(parents=True, exist_ok=True)
    marker = markers / f'{archive.name}.sha256'
    if marker.exists() and marker.read_text(encoding='ascii').strip() == observed:
        print('Hazır:', archive.name, observed[:12])
        return
    root = destination.resolve()
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            if not (destination / member.filename).resolve().is_relative_to(root):
                raise RuntimeError(f'Güvensiz ZIP yolu: {member.filename}')
        bundle.extractall(destination)
    marker.write_text(observed, encoding='ascii')
    print('Açıldı:', archive.name, observed[:12])
for archive, observed in verified:
    extract_verified(archive, observed, REPO_ROOT)
RUNNER = REPO_ROOT / 'scripts/four_dataset_probabilistic_gpu_v2_runner.py'
if not RUNNER.is_file():
    raise FileNotFoundError(f'Runner bulunamadı: {RUNNER}')

## 4. Bağımlılıklar ve contract verification

Verify başarısızsa eğitime geçmeyin; özellikle holdout izolasyonu ve normal-only optimizer rolü korunmalıdır.

In [ ]:
import os
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'pyarrow',
                'scikit-learn', 'scipy', 'matplotlib'], check=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_ROOT)
def runner_command(action):
    return [sys.executable, str(RUNNER), action, '--repo-root', str(REPO_ROOT),
            '--dataset', DATASET, '--run-dir', str(RUN_DIR), '--device', 'cuda']
verify_cmd = runner_command('verify')
print(' '.join(verify_cmd))
subprocess.run(verify_cmd, check=True, env=env)
print('VERIFY PASS:', DATASET)

## 5. Resumable train

Aynı Drive `RUN_DIR` ile tekrar çalıştırıldığında runner checkpoint'ten devam eder. Colab koparsa önceki hücreleri yeniden çalıştırın; frozen değerleri değiştirmeyin.

In [ ]:
train_cmd = runner_command('train')
print(' '.join(train_cmd))
subprocess.run(train_cmd, check=True, env=env)
print('TRAIN tamamlandı:', RUN_DIR)

## 6. Checkpoint, history ve report incelemesi

In [ ]:
import json
run_files = sorted(p for p in RUN_DIR.rglob('*') if p.is_file())
for path in run_files:
    print(path.relative_to(RUN_DIR), f'{path.stat().st_size / 2**20:.2f} MiB')
checkpoint_files = sorted(set(RUN_DIR.glob('*checkpoint*.pt')) | set(RUN_DIR.glob('*checkpoint*.pth')))
for path in checkpoint_files:
    payload = torch.load(path, map_location='cpu', weights_only=False)
    print('\nCHECKPOINT', path.name, '| keys:', sorted(payload) if isinstance(payload, dict) else type(payload).__name__)
    if isinstance(payload, dict):
        for key in ('completed_epochs', 'epoch', 'epoch_index', 'batch_index', 'source_index', 'global_step', 'saved_at_utc'):
            if key in payload:
                print(key, ':', payload[key])
json_files = sorted(set(RUN_DIR.glob('*history*.json')) | set(RUN_DIR.glob('*report*.json')) | set(RUN_DIR.glob('*summary*.json')) | set(RUN_DIR.glob('*progress*.json')))
for path in json_files:
    print('\nJSON', path.name)
    print(json.dumps(json.loads(path.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
if not checkpoint_files and not json_files:
    print('Henüz checkpoint/history/report yok; train çıktısını kontrol edin.')

## 7. Basit loss grafiği

Grafik yalnız izleme içindir; tuning amacıyla kullanılmaz.

In [ ]:
import matplotlib.pyplot as plt
def records_from(value):
    if isinstance(value, list) and all(isinstance(x, dict) for x in value):
        return value
    if isinstance(value, dict):
        for key in ('history', 'epochs', 'training_history'):
            if isinstance(value.get(key), list):
                return value[key]
    return []
payloads = [json.loads(p.read_text(encoding='utf-8')) for p in
            sorted(set(RUN_DIR.glob('*history*.json')) | set(RUN_DIR.glob('*report*.json')))]
if checkpoint_files:
    payloads.append(torch.load(checkpoint_files[-1], map_location='cpu', weights_only=False))
records = next((r for value in payloads if (r := records_from(value))), [])
def number(record, keys):
    return next((float(record[k]) for k in keys if isinstance(record.get(k), (int, float))), None)
epochs = [r.get('epoch', i + 1) for i, r in enumerate(records)]
train = [number(r, ('mean_train_gaussian_nll', 'train_loss', 'train_nll', 'mean_train_nll', 'mean_weighted_gaussian_nll', 'loss')) for r in records]
valid = [number(r, ('mean_validation_gaussian_nll', 'val_loss', 'validation_loss', 'val_nll', 'mean_val_nll')) for r in records]
if records and any(v is not None for v in train + valid):
    plt.figure(figsize=(8, 4))
    if any(v is not None for v in train): plt.plot(epochs, train, marker='o', label='train')
    if any(v is not None for v in valid): plt.plot(epochs, valid, marker='o', label='validation')
    plt.xlabel('Epoch'); plt.ylabel('Masked Gaussian NLL / loss')
    plt.title(f'{DATASET} probabilistic GPU v1'); plt.grid(alpha=.3); plt.legend(); plt.show()
else:
    print('Çizilebilir history henüz yok; keys:', sorted(records[0]) if records else [])

`training_report.json` içindeki magnitude diagnostic development sonucu olduğu gibi raporlanır. `rho >= 0.8` magnitude-dominated işaretini iyileştirmek için aynı v2 ayarları değiştirilmez; blind holdout hiçbir hücrede açılmaz.